# Assignment 5 — Catching Data Leakage Before It Catches You

**Course:** Feature Engineering & MLOps  
**Topic:** Data Leakage — Target Leakage and Preprocessing Leakage

This notebook completes Sections **4.1–4.3** of Assignment 5 using `data/raw/customer_churn_a5.csv`. The analysis demonstrates target leakage, preprocessing leakage, and the corrected deployable pipeline.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42
TEST_SIZE = 0.25
DATA_PATH = "../data/raw/customer_churn_a5.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (600, 8)


,customer_id,tenure_months,monthly_charges,contract_type,support_calls,churn,days_since_cancellation,final_bill_amount
0,20406,13,99.17,Month-to-month,2,0,NaN,NaN
1,20531,24,66.77,Month-to-month,1,0,NaN,NaN
2,20307,15,44.03,One year,1,0,NaN,NaN
3,20391,3,61.08,Two year,3,0,NaN,NaN
4,20147,8,106.72,Two year,1,0,NaN,NaN


## 3. Dataset Overview

The assignment specifies 600 telecom customers. The legitimate predictors are `tenure_months`, `monthly_charges`, `contract_type`, and `support_calls`. The columns `days_since_cancellation` and `final_bill_amount` are intentionally leaked because they only exist after cancellation. `churn` is the target.

In [2]:
legitimate_features = ["tenure_months", "monthly_charges", "contract_type", "support_calls"]
leaked_features = ["days_since_cancellation", "final_bill_amount"]
numeric_legitimate_features = ["tenure_months", "monthly_charges", "support_calls"]
target = "churn"

print("Rows:", len(df))
print("Target counts:")
print(df[target].value_counts().sort_index())
print("\nMissing values:")
print(df.isna().sum())

Rows: 600
Target counts:
churn
0    451
1    149
Name: count, dtype: int64

Missing values:
customer_id                  0
tenure_months                0
monthly_charges              0
contract_type                0
support_calls                0
churn                        0
days_since_cancellation    451
final_bill_amount          451
dtype: int64


# 4.1 — Target leakage: prove it, don't just assert it

A target-leaked feature should fail the question: **“Could this feature exist at prediction time?”** Both cancellation-related columns are only available after churn has already occurred, so they cannot be legitimate predictors for a new customer at scoring time.

In [3]:
non_null_cancellation = df["days_since_cancellation"].notna().sum()
churn_rows = (df["churn"] == 1).sum()

print("Non-null days_since_cancellation:", non_null_cancellation)
print("Number of churn=1 rows:", churn_rows)
print("Exact equality:", non_null_cancellation == churn_rows)

Non-null days_since_cancellation: 149
Number of churn=1 rows: 149
Exact equality: True


In [4]:
final_bill_filled = df["final_bill_amount"].fillna(-1)
final_bill_corr = final_bill_filled.corr(df["churn"])
tenure_corr = df["tenure_months"].corr(df["churn"])

correlation_table = pd.DataFrame({
    "feature": ["final_bill_amount (filled -1)", "tenure_months"],
    "correlation_with_churn": [final_bill_corr, tenure_corr]
})
correlation_table

,feature,correlation_with_churn
0,final_bill_amount (filled -1),0.924978
1,tenure_months,-0.220718


### Model setup

A `LogisticRegression` classifier is used for the required accuracy and ROC-AUC comparison. The train/test split is fixed at 75%/25% with `random_state=42` and stratification on `churn`. The customer ID is not used because it is an identifier rather than a meaningful predictor.

In [5]:
def build_model(feature_list):
    numeric_features = [c for c in feature_list if c != "contract_type"]
    categorical_features = ["contract_type"] if "contract_type" in feature_list else []

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ])

X = df[legitimate_features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

model_a = build_model(legitimate_features)
model_a.fit(X_train, y_train)
model_a_pred = model_a.predict(X_test)
model_a_prob = model_a.predict_proba(X_test)[:, 1]

model_a_accuracy = accuracy_score(y_test, model_a_pred)
model_a_auc = roc_auc_score(y_test, model_a_prob)

print(f"Model A test accuracy: {model_a_accuracy:.6f}")
print(f"Model A test ROC-AUC:  {model_a_auc:.6f}")

Model A test accuracy: 0.753333
Model A test ROC-AUC:  0.778283


In [6]:
X_leaked = df[legitimate_features + leaked_features].copy()
X_leaked[leaked_features] = X_leaked[leaked_features].fillna(-1)

X_train_leaked, X_test_leaked, y_train_leaked, y_test_leaked = train_test_split(
    X_leaked, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

model_b = build_model(legitimate_features + leaked_features)
model_b.fit(X_train_leaked, y_train_leaked)
model_b_pred = model_b.predict(X_test_leaked)
model_b_prob = model_b.predict_proba(X_test_leaked)[:, 1]

model_b_accuracy = accuracy_score(y_test_leaked, model_b_pred)
model_b_auc = roc_auc_score(y_test_leaked, model_b_prob)

print(f"Model B test accuracy: {model_b_accuracy:.6f}")
print(f"Model B test ROC-AUC:  {model_b_auc:.6f}")

Model B test accuracy: 1.000000
Model B test ROC-AUC:  1.000000


In [7]:
accuracy_gap = model_b_accuracy - model_a_accuracy
auc_gap = model_b_auc - model_a_auc

comparison = pd.DataFrame({
    "model": ["Model A — legitimate features", "Model B — legitimate + leaked features"],
    "test_accuracy": [model_a_accuracy, model_b_accuracy],
    "roc_auc": [model_a_auc, model_b_auc]
})
comparison["accuracy_gain_vs_A"] = comparison["test_accuracy"] - model_a_accuracy
comparison["auc_gain_vs_A"] = comparison["roc_auc"] - model_a_auc
comparison

print(f"Accuracy gap (B - A): {accuracy_gap:.6f}")
print(f"ROC-AUC gap (B - A):  {auc_gap:.6f}")

Accuracy gap (B - A): 0.246667
ROC-AUC gap (B - A):  0.221717


**Interpretation:** Model B reaches a perfect-looking test result because the leaked variables contain information that is only created after the cancellation decision. A production system scoring a brand-new customer would not have `days_since_cancellation` or `final_bill_amount` available at prediction time, so the model could not reproduce the same inputs or its apparent 1.00 performance. The result is an example of an unrealistically optimistic evaluation caused by target leakage.

# 4.2 — Preprocessing leakage: the wrong order vs. the right order

In [8]:
# WRONG ORDER: fit the scaler on the entire dataset before splitting.
full_scaler = StandardScaler()
full_scaler.fit(df[numeric_legitimate_features])

print("Full-dataset scaler mean_:", full_scaler.mean_)
print("Full-dataset scaler scale_:", full_scaler.scale_)

Full-dataset scaler mean_: [19.56166667 65.92201667  2.165     ]
Full-dataset scaler scale_: [17.52454081 22.79931919  1.49814608]


In [9]:
# RIGHT ORDER: split first, then fit a NEW scaler on training data only.
X_num = df[numeric_legitimate_features].copy()
X_num_train, X_num_test, y_num_train, y_num_test = train_test_split(
    X_num, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

train_scaler = StandardScaler()
train_scaler.fit(X_num_train)

print("Training-only scaler mean_:", train_scaler.mean_)
print("Training-only scaler scale_:", train_scaler.scale_)

Training-only scaler mean_: [19.89555556 65.79473333  2.11777778]
Training-only scaler scale_: [17.44515795 22.26390441  1.48380725]


In [10]:
mean_difference = full_scaler.mean_ - train_scaler.mean_
mean_difference_abs = np.abs(mean_difference)

scaler_comparison = pd.DataFrame({
    "feature": numeric_legitimate_features,
    "full_data_mean": full_scaler.mean_,
    "train_only_mean": train_scaler.mean_,
    "mean_difference": mean_difference,
    "absolute_difference": mean_difference_abs
})
scaler_comparison

,feature,full_data_mean,train_only_mean,mean_difference,absolute_difference
0,tenure_months,19.561667,19.895556,-0.333889,0.333889
1,monthly_charges,65.922017,65.794733,0.127283,0.127283
2,support_calls,2.165000,2.117778,0.047222,0.047222


**Why the principle matters:** Even a small difference means the transformation learned information from the held-out test set. The test set is supposed to represent unseen future data, so allowing its distribution to influence preprocessing makes the evaluation less independent. On a different or smaller dataset, the difference can be much larger and can change model selection decisions.

In [11]:
# Correct single Pipeline: imputer + scaler + model, fit only on the training split.
numeric_only_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

numeric_only_pipeline.fit(X_num_train, y_num_train)
pipeline_scaler = numeric_only_pipeline.named_steps["scaler"]

print("Pipeline scaler mean_:", pipeline_scaler.mean_)
print("Pipeline scaler scale_:", pipeline_scaler.scale_)
print("Means match manual correct scaler:", np.allclose(pipeline_scaler.mean_, train_scaler.mean_))
print("Scales match manual correct scaler:", np.allclose(pipeline_scaler.scale_, train_scaler.scale_))

Pipeline scaler mean_: [19.89555556 65.79473333  2.11777778]
Pipeline scaler scale_: [17.44515795 22.26390441  1.48380725]
Means match manual correct scaler: True
Scales match manual correct scaler: True


# 4.3 — The fix

The final deployable model uses only legitimate features and keeps all preprocessing inside a single sklearn `Pipeline`. The train/test split occurs first, and the pipeline is fitted only on `X_train`, so the preprocessing cannot learn from the test set.

In [12]:
final_model = build_model(legitimate_features)
final_model.fit(X_train, y_train)

final_pred = final_model.predict(X_test)
final_prob = final_model.predict_proba(X_test)[:, 1]

final_accuracy = accuracy_score(y_test, final_pred)
final_auc = roc_auc_score(y_test, final_prob)

print(f"Final fixed model test accuracy: {final_accuracy:.6f}")
print(f"Final fixed model test ROC-AUC:  {final_auc:.6f}")
print(f"Matches Model A accuracy: {np.isclose(final_accuracy, model_a_accuracy)}")
print(f"Matches Model A ROC-AUC:  {np.isclose(final_auc, model_a_auc)}")

Final fixed model test accuracy: 0.753333
Final fixed model test ROC-AUC:  0.778283
Matches Model A accuracy: True
Matches Model A ROC-AUC:  True


In [13]:
final_summary = pd.DataFrame({
    "metric": ["Test Accuracy", "ROC-AUC"],
    "Model A (honest baseline)": [model_a_accuracy, model_a_auc],
    "Final fixed model": [final_accuracy, final_auc],
    "difference": [final_accuracy - model_a_accuracy, final_auc - model_a_auc]
})
final_summary

,metric,Model A (honest baseline),Final fixed model,difference
0,Test Accuracy,0.753333,0.753333,0.0
1,ROC-AUC,0.778283,0.778283,0.0


## Conclusion

The analysis demonstrates both required leakage mechanisms. The target-leaked variables are directly tied to the post-cancellation outcome: `days_since_cancellation` is non-null exactly for the 149 churned customers, and `final_bill_amount` has a much stronger correlation with churn than the legitimate `tenure_months` feature. Adding those leaked variables raises the test accuracy and ROC-AUC from the honest baseline of **0.753333 / 0.778283** to **1.000000 / 1.000000**, which is not a deployable result. Preprocessing leakage is also visible because the scaler fitted on all 600 rows learns different means from the scaler fitted on the 450-row training split. The final pipeline prevents both problems by excluding post-outcome features and fitting preprocessing only after the train/test split.

# 7. Bonus (Optional, +10% extra credit)

Even cross-validation can leak when preprocessing is fitted before the CV split. The comparison below uses the three legitimate numeric features and 5-fold stratified cross-validation.

In [14]:
from sklearn.pipeline import make_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# WRONG: scaler sees all rows before the CV folds are formed.
cv_scaler = StandardScaler()
X_num_scaled = cv_scaler.fit_transform(X_num)
wrong_cv_scores = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    X_num_scaled,
    y,
    cv=cv,
    scoring="roc_auc"
)

# RIGHT: scaler is fitted separately inside every training fold.
correct_cv_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
)
right_cv_scores = cross_val_score(
    correct_cv_pipeline,
    X_num,
    y,
    cv=cv,
    scoring="roc_auc"
)

bonus_comparison = pd.DataFrame({
    "fold": range(1, 6),
    "leaky_cv_auc": wrong_cv_scores,
    "pipeline_cv_auc": right_cv_scores,
    "difference": wrong_cv_scores - right_cv_scores
})
bonus_comparison.loc["mean"] = [
    "Mean", wrong_cv_scores.mean(), right_cv_scores.mean(),
    wrong_cv_scores.mean() - right_cv_scores.mean()
]
bonus_comparison

,fold,leaky_cv_auc,pipeline_cv_auc,difference
0,1,0.666540,0.666540,0.000000
1,2,0.712593,0.712593,0.000000
2,3,0.714815,0.714815,0.000000
3,4,0.650000,0.649259,0.000741
4,5,0.620000,0.620000,0.000000
mean,Mean,0.672790,0.672641,0.000148


**Bonus interpretation:** The two CV approaches are extremely close on this dataset (mean ROC-AUC approximately **0.672790** for the leaky version versus **0.672641** for the proper pipeline). A small difference does not make the wrong procedure valid: the whole-dataset scaler has still used information from every fold before validation. Keeping transformations inside the pipeline is the safer and reproducible rule because each CV training fold learns its own preprocessing parameters.